[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/13_beyond_accuracy/13_proyecto_reranker_home.ipynb)

# Proyecto 13 · Re-ranker de la home de CineMatch: relevancia + diversidad + calibración

**Nivel:** 🟠 Avanzado → 🔴 Experto · **Duración:** 4–6 h · **GPU:** no necesaria · **Unidades:** ~0–1.
**Prerrequisitos:** lección 13 (y 01, 02, 08).

## 🏢 Contexto de negocio
El equipo de producto de **CineMatch** tiene tres quejas sobre la home:
1. *"Me salen 10 películas del mismo tipo"* (falta de **diversidad**).
2. *"Veo con mis hijos dibujos y con mi pareja thrillers, pero solo me recomienda thrillers"* (falta de **calibración**).
3. *"La misma película aparece en tres filas"* y los perfiles infantiles han visto un tráiler de terror (**página** y **reglas**).

El VP pide un re-ranker que arregle esto **sin perder más de un 7 % de NDCG@10 offline** y un análisis de qué pasará con la concentración del catálogo a medio plazo.

## 📦 Dataset
**MovieLens latest-small** (GroupLens). Géneros como espacio de diversidad/calibración. Candidatos: top-100 de un retriever PureSVD + FAISS (módulo 08). Si no hay red, datos sintéticos con la misma forma.

## ✅ Entregables y rúbrica (sobre usuarios de **test**, pesos ajustados en usuarios de **validación**)

| # | Entregable | Criterio |
|---|---|---|
| R1 | Métricas `ild`, `kl_calibration`, `gini` | pasan los tests numéricos |
| R2 | `dpp_greedy_fast` (Cholesky incremental) | coincide con el greedy ingenuo |
| R3 | `rerank_home(u, ids, rel, w)` (diversidad + calibración + novedad) | NDCG@10 ≥ 0,93 × baseline · ILD@10 ≥ 1,10 × baseline · KL ≤ 0,75 × baseline |
| R4 | Barrido de pesos + frente de Pareto + punto elegido por ε-restricción en **validación** | gráfico y tabla |
| R5 | `build_home(u)`: 6 filas × 10 títulos, deduplicada, con reglas (infantil, cuota por género) | 0 duplicados · 0 violaciones en perfiles infantiles · utilidad ≥ filas independientes |
| R6 | Simulación de feedback loop (10 rondas): baseline vs tu re-ranker | gráfico de Gini y cobertura |

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import os, re, json, time, math, random, itertools, collections, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True                      # True: muestra de usuarios y menos rondas de simulación
DATASET = "ml-latest-small"              # o "ml-1m" (6.040 usuarios) con FAST_DEV_RUN=False
K = 10                                   # longitud de la lista final
N_CAND = 100                             # candidatos que llegan al re-ranker
N_EVAL_USERS = 300 if FAST_DEV_RUN else 3000
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Este módulo es 100 % CPU: no necesitas GPU.")

### Utilidades (módulos 01, 02, 08) y preparación de candidatos

In [ ]:
# ==== Utilidades mínimas (vienen del módulo 01: datos) ====
import io, zipfile, urllib.request
from pathlib import Path

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
ML_URLS = {
    "ml-latest-small": "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip",
    "ml-1m": "https://files.grouplens.org/datasets/movielens/ml-1m.zip",
}
ML_GENRES = ["Action", "Adventure", "Animation", "Children", "Comedy", "Crime", "Documentary",
             "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
             "Sci-Fi", "Thriller", "War", "Western"]

def _download_movielens(variant: str) -> Path:
    zpath = DATA_DIR / f"{variant}.zip"
    if not zpath.exists():
        urllib.request.urlretrieve(ML_URLS[variant], zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_DIR)
    return DATA_DIR / variant

def make_synthetic_movielens(n_users=600, n_items=1500, seed=42):
    """Fallback sintético con la MISMA forma que MovieLens (por si no hay red)."""
    rng = np.random.default_rng(seed)
    words = {g: [f"{g[:4]}{k}" for k in range(6)] for g in ML_GENRES}
    rows_m = []
    for i in range(n_items):
        gs = list(rng.choice(ML_GENRES, size=rng.integers(1, 4), replace=False))
        year = int(rng.integers(1950, 2019))
        title = " ".join(rng.choice(words[gs[0]], 2)) + f" {i} ({year})"
        rows_m.append((i, title, "|".join(gs)))
    movies = pd.DataFrame(rows_m, columns=["item_id", "title", "genres"])
    G = np.array([[g in m.split("|") for g in ML_GENRES] for m in movies.genres], float)
    pop = rng.zipf(1.6, n_items).clip(1, 500).astype(float)
    rows_r = []
    for u in range(n_users):
        taste = rng.dirichlet(np.full(len(ML_GENRES), 0.2))
        aff = G @ taste / G.sum(1)
        p = pop ** 0.6 * np.exp(10 * aff); p /= p.sum()
        n = int(rng.integers(20, 150))
        items = rng.choice(n_items, size=n, replace=False, p=p)
        rat = np.clip(np.round(2.5 + 8 * aff[items] + rng.normal(0, .7, n)), 1, 5)
        ts = np.sort(rng.integers(9e8, 1.5e9, n))
        rows_r += list(zip([u] * n, items, rat, ts))
    ratings = pd.DataFrame(rows_r, columns=["user_id", "item_id", "rating", "timestamp"])
    return ratings, movies

In [ ]:
def load_movielens(variant: str = "ml-latest-small"):
    """Devuelve (ratings[user_id,item_id,rating,timestamp], movies[item_id,title,genres,year]) con ids contiguos."""
    try:
        d = _download_movielens(variant)
        if variant == "ml-1m":
            ratings = pd.read_csv(d / "ratings.dat", sep="::", engine="python",
                                  names=["user_id", "item_id", "rating", "timestamp"])
            movies = pd.read_csv(d / "movies.dat", sep="::", engine="python", encoding="latin-1",
                                 names=["item_id", "title", "genres"])
        else:
            ratings = pd.read_csv(d / "ratings.csv").rename(columns={"userId": "user_id", "movieId": "item_id"})
            movies = pd.read_csv(d / "movies.csv").rename(columns={"movieId": "item_id"})
        print(f"MovieLens {variant} descargado de GroupLens")
    except Exception as e:  # sin red / URL caída → nunca bloqueamos el notebook
        print(f"⚠️ No se pudo descargar MovieLens ({type(e).__name__}). Uso datos SINTÉTICOS con la misma forma.")
        ratings, movies = make_synthetic_movielens()
    movies = movies[movies.item_id.isin(ratings.item_id.unique())].reset_index(drop=True)
    item_map = {old: new for new, old in enumerate(movies.item_id)}
    user_map = {old: new for new, old in enumerate(sorted(ratings.user_id.unique()))}
    ratings = ratings.assign(item_id=ratings.item_id.map(item_map), user_id=ratings.user_id.map(user_map))
    movies = movies.assign(item_id=movies.item_id.map(item_map))
    movies["year"] = movies.title.str.extract(r"\((\d{4})\)\s*$")[0].astype(float)
    movies["genre_list"] = movies.genres.str.replace("Children's", "Children").str.split("|")   # ml-1m → mismo nombre
    return ratings.sort_values(["user_id", "timestamp"]).reset_index(drop=True), movies

In [ ]:
# ==== Utilidades mínimas (vienen de los módulos 01 y 02: split temporal + métricas) ====
def split_temporal_per_user(ratings: pd.DataFrame, test_frac: float = 0.2, min_train: int = 5):
    """Para cada usuario, el último `test_frac` de sus interacciones (por tiempo) va a test."""
    r = ratings.sort_values(["user_id", "timestamp"]).copy()
    r["rank_t"] = r.groupby("user_id").cumcount()
    r["n_u"] = r.groupby("user_id")["item_id"].transform("size")
    n_test = np.maximum(1, np.floor(r["n_u"] * test_frac)).astype(int)
    is_test = (r["rank_t"] >= r["n_u"] - n_test) & (r["n_u"] > min_train)
    return r[~is_test].drop(columns=["rank_t", "n_u"]), r[is_test].drop(columns=["rank_t", "n_u"])

def recall_at_k(recs, truth, k=10):
    return len(set(recs[:k]) & truth) / max(1, min(k, len(truth)))

def ndcg_at_k(recs, truth, k=10):
    dcg = sum(1 / np.log2(i + 2) for i, it in enumerate(recs[:k]) if it in truth)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(truth))))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(recs_by_user: dict, truth_by_user: dict, k: int = 10) -> dict:
    us = [u for u in recs_by_user if u in truth_by_user and truth_by_user[u]]
    return {f"Recall@{k}": float(np.mean([recall_at_k(recs_by_user[u], truth_by_user[u], k) for u in us])),
            f"NDCG@{k}": float(np.mean([ndcg_at_k(recs_by_user[u], truth_by_user[u], k) for u in us]))}

In [ ]:
# ==== Utilidades mínimas (vienen del módulo 08: retriever de embeddings + FAISS) ====
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
try:
    import faiss
except ImportError:
    faiss = None

def build_interaction_matrix(train: pd.DataFrame, n_users: int, n_items: int):
    return sp.csr_matrix((np.ones(len(train), np.float32), (train.user_id, train.item_id)),
                         shape=(n_users, n_items))

class EmbeddingRetriever:
    """Top-K por producto interno con FAISS (IndexFlatIP); fallback NumPy si no hay FAISS."""
    def __init__(self, item_emb: np.ndarray):
        self.item_emb = np.ascontiguousarray(item_emb, dtype=np.float32)
        self.index = None
        if faiss is not None:
            self.index = faiss.IndexFlatIP(self.item_emb.shape[1]); self.index.add(self.item_emb)

    def search(self, q: np.ndarray, k: int = 100, exclude=None):
        q = np.ascontiguousarray(np.atleast_2d(q), dtype=np.float32)
        extra = max((len(e) for e in exclude), default=0) if exclude is not None else 0
        kk = min(k + extra, len(self.item_emb))
        if self.index is not None:
            scores, ids = self.index.search(q, kk)
        else:
            s = q @ self.item_emb.T
            ids = np.argsort(-s, axis=1)[:, :kk]; scores = np.take_along_axis(s, ids, 1)
        out = []
        for row, (sc, ii) in enumerate(zip(scores, ids)):
            ex = exclude[row] if exclude is not None else set()
            keep = [(int(i), float(s)) for i, s in zip(ii, sc) if i >= 0 and i not in ex][:k]
            out.append(keep)
        return out

class PureSVDRecommender:
    """CF ligero (PureSVD, Cremonesi 2010) como sustituto del two-tower del módulo 08."""
    def __init__(self, X: sp.csr_matrix, dim: int = 64, seed: int = 42):
        self.X = X
        svd = TruncatedSVD(n_components=dim, random_state=seed).fit(X)
        self.item_emb = svd.components_.T.astype(np.float32)      # V  (n_items × d)
        self.retriever = EmbeddingRetriever(self.item_emb)

    def user_emb(self, users):
        return np.asarray(self.X[users] @ self.item_emb)          # u = x_u V

    def recommend(self, users, k=10, exclude_seen=True):
        users = list(users)
        excl = [set(self.X[u].indices) for u in users] if exclude_seen else None
        res = self.retriever.search(self.user_emb(users), k, excl)
        return {u: [i for i, _ in r] for u, r in zip(users, res)}

In [ ]:
ratings, movies = load_movielens(DATASET)
n_users, n_items = ratings.user_id.nunique(), len(movies)
train, test = split_temporal_per_user(ratings, test_frac=0.2)
X = build_interaction_matrix(train, n_users, n_items)
pop = np.asarray(X.sum(0)).ravel()
truth = test.groupby("user_id").item_id.apply(set).to_dict()
hist_by_user = train.groupby("user_id").item_id.apply(list).to_dict()

GENRES = sorted({g for gs in movies.genre_list for g in gs if g in ML_GENRES})
G = np.array([[g in gs for g in GENRES] for gs in movies.genre_list], dtype=np.float32)
G[G.sum(1) == 0, GENRES.index("Drama")] = 1                      # ítems sin género → Drama
P_gi = G / G.sum(1, keepdims=True)                                # p(g|i) uniforme entre sus géneros
G_unit = G / np.linalg.norm(G, axis=1, keepdims=True)
SIM = lambda ids: G_unit[ids] @ G_unit[ids].T                     # similitud coseno por géneros

cf = PureSVDRecommender(X, dim=64)
rng = np.random.default_rng(SEED)
eval_users = sorted(int(u) for u in rng.choice(sorted(truth), size=min(N_EVAL_USERS, len(truth)), replace=False))

def candidates(u, n=N_CAND):
    """Top-n del retriever con su puntuación de relevancia normalizada a [0, 1]."""
    ids, sc = zip(*cf.retriever.search(cf.user_emb([u]), n, [set(hist_by_user[u])])[0])
    sc = np.array(sc); sc = (sc - sc.min()) / (sc.max() - sc.min() + 1e-9)
    return np.array(ids), sc

CANDS = {u: candidates(u) for u in eval_users}
print(f"{n_users} usuarios · {n_items} películas · {len(GENRES)} géneros · {len(eval_users)} usuarios de evaluación")

In [ ]:
# Usuarios de VALIDACIÓN (para ajustar pesos) disjuntos de los de TEST (eval_users)
rest = sorted(set(truth) - set(eval_users))
val_users = sorted(int(u) for u in np.random.default_rng(1).choice(rest, size=min(len(eval_users), len(rest)), replace=False))
for u in val_users:
    CANDS[u] = candidates(u)
NOV = (lambda v: (v - v.min()) / (v.max() - v.min() + 1e-9))(-np.log2((pop + 1) / (n_users + 1)))

def genre_dist_history(u):
    h = hist_by_user[u]; w = np.linspace(.5, 1., len(h))
    p = (w[:, None] * P_gi[h]).sum(0); return p / p.sum()

def genre_dist_list(ids):
    q = P_gi[list(ids)].sum(0); return q / max(q.sum(), 1e-12)

def check(name, fn):
    try:
        fn(); print(f"✅ {name}")
    except NotImplementedError:
        print(f"⏳ {name}: TODO pendiente")
    except AssertionError as e:
        print(f"❌ {name}: {e}")
print(f"test={len(eval_users)} usuarios · validación={len(val_users)} usuarios")

## 🛠️ Parte 1 · Métricas (R1)
- `ild(ids)`: media de $1 - \cos$ (géneros, usa `SIM(ids)`) sobre pares $i<j$.
- `kl_calibration(p, q, alpha=0.01)`: $\mathrm{KL}(p\Vert\tilde q)$ con $\tilde q = (1-\alpha)q + \alpha p$, sumando solo donde $p>0$.
- `gini(x)`: coeficiente de Gini de un vector de exposiciones.

In [ ]:
def ild(ids):
    raise NotImplementedError   # TODO

def kl_calibration(p, q, alpha=0.01):
    raise NotImplementedError   # TODO

def gini(x):
    raise NotImplementedError   # TODO

def _t1():
    assert abs(kl_calibration(np.array([.7, .3]), np.array([1., 0.])) - 1.1336) < 1e-3, "KL del ejemplo 70/30"
    assert abs(gini(np.ones(10))) < 1e-9 and gini(np.r_[np.zeros(9), 1]) > .89, "Gini"
    ids = np.where(G[:, GENRES.index("Drama")] > 0)[0][:3]
    assert ild(ids) >= 0 and abs(ild([ids[0], ids[0]])) < 1e-6, "ILD"
check("R1 métricas", _t1)

## 🛠️ Parte 2 · DPP rápido (R2)
Implementa el greedy MAP de Chen et al. (2018): mantén para cada candidato el vector de Cholesky $\mathbf{c}_i$ y $d_i^2$; al elegir $j$, $e_i = (L_{ji} - \langle \mathbf{c}_j, \mathbf{c}_i\rangle)/d_j$, $d_i^2 \mathrel{-}= e_i^2$. Para cuando $\max d_i^2 < \varepsilon$.

In [ ]:
def dpp_greedy_fast(L, k, eps=1e-10):
    raise NotImplementedError   # TODO

def dpp_greedy_naive(L, k):
    chosen = []
    for _ in range(k):
        cand = [(np.linalg.slogdet(L[np.ix_(chosen + [i], chosen + [i])]), i) for i in range(len(L)) if i not in chosen]
        (sign, v), i = max(((s, v), i) for (s, v), i in cand if s > 0)
        chosen.append(i)
    return chosen

def _t2():
    g = np.random.default_rng(0); B = g.normal(size=(20, 80)); L = B.T @ B / 20 + 1e-3 * np.eye(80)
    assert dpp_greedy_fast(L, 8) == dpp_greedy_naive(L, 8), "no coincide con el greedy ingenuo"
check("R2 DPP", _t2)

## 🛠️ Parte 3 · Re-ranker de la home (R3)
`rerank_home(u, ids, rel, w)` con `w = {"div": ..., "cal": ..., "nov": ...}`. Greedy: en cada paso elige el candidato que maximiza
$$\operatorname{rel}(i) + w_{div}(1-\max_{j\in S}\operatorname{sim}(i,j)) + w_{nov}\operatorname{nov}(i) - w_{cal}\,\mathrm{KL}(p\Vert \tilde q(S\cup\{i\})).$$
💡 Vectoriza la KL para todos los candidatos a la vez (una matriz `Q` de tamaño $N \times |\mathcal{G}|$).

In [ ]:
def rerank_home(u, ids, rel, w, k=K):
    raise NotImplementedError   # TODO

def evaluate_lists(lists, k=K):
    us = list(lists); exp = np.zeros(n_items)
    for u in us:
        exp[list(lists[u][:k])] += 1
    return {"NDCG@10": np.mean([ndcg_at_k(list(lists[u]), truth[u], k) for u in us]),
            "ILD@10": np.mean([ild(lists[u][:k]) for u in us]),
            "KL": np.mean([kl_calibration(genre_dist_history(u), genre_dist_list(lists[u][:k])) for u in us]),
            "Novedad": np.mean([NOV[list(lists[u][:k])].mean() for u in us]),
            "Cobertura": float((exp > 0).mean()), "Gini": gini(exp)}

## 🛠️ Parte 4 · Pareto + ε-restricción en validación (R4)
Barre al menos 30 combinaciones de pesos en `val_users`, marca las no dominadas en (NDCG, ILD, −KL) y elige por **ε-restricción**: entre las que cumplen NDCG ≥ 0,93×, ILD ≥ 1,10× y KL ≤ 0,75× el baseline **de validación**, la de mayor NDCG (si ninguna cumple, la de Pareto con mejor compromiso y NDCG ≥ 0,93×). Después evalúa ese punto **una sola vez** en test.

In [ ]:
def pareto_mask(P):
    raise NotImplementedError   # TODO (todas las columnas se maximizan)

def choose_weights(grid, users):
    raise NotImplementedError   # TODO: devuelve (DataFrame del barrido, dict de pesos elegidos)

## 🛠️ Parte 5 · Home completa con reglas (R5)
`build_home(u, kids=False)` devuelve una lista de 6 filas `(titulo_fila, [10 ids])`:
- Filas candidatas: "Top para ti" (tu `rerank_home`), 3 × "Porque te gusta <género>", "Tendencias", "Joyas ocultas".
- **Greedy de página** con el modelo de atención `ATT` y deduplicación.
- **Reglas**: en perfil infantil se eliminan `MATURE`; ninguna fila con más de 4 títulos del mismo género principal.

In [ ]:
N_ROWS, N_COLS, VISIBLE = 6, 10, 5
ATT = (0.8 ** np.arange(N_ROWS))[:, None] * np.where(np.arange(N_COLS) < VISIBLE, 1., .6 ** (np.arange(N_COLS) - VISIBLE + 1))[None, :]
MATURE = {"Horror", "Thriller", "Crime", "War", "Film-Noir"}

def build_home(u, kids=False):
    raise NotImplementedError   # TODO

def page_value(page, r):
    seen, util, dups = set(), 0.0, 0
    for pos, (_, items) in enumerate(page):
        for c, i in enumerate(items):
            if i in seen:
                dups += 1
            else:
                seen.add(i); util += ATT[pos, c] * r[i]
    return util, dups

---
# ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa: redefine las funciones y ejecuta la evaluación final (R1–R6).

In [ ]:
# ---------- R1 ----------
def ild(ids):
    ids = list(ids)
    if len(ids) < 2:
        return 0.0
    S = SIM(ids); return float((1 - S)[np.triu_indices(len(ids), 1)].mean())

def kl_calibration(p, q, alpha=0.01):
    qt = (1 - alpha) * q + alpha * p; m = p > 0
    return float(np.sum(p[m] * np.log(p[m] / qt[m])))

def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return 0.0 if x.sum() == 0 else float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))

check("R1 métricas", _t1)

In [ ]:
# ---------- R2 ----------
def dpp_greedy_fast(L, k, eps=1e-10):
    N = L.shape[0]; c = np.zeros((k, N)); d2 = np.diag(L).astype(float).copy()
    chosen = [int(np.argmax(d2))]
    while len(chosen) < k:
        j, t = chosen[-1], len(chosen) - 1
        e = (L[j] - c[:t, j] @ c[:t]) / np.sqrt(d2[j])
        c[t] = e; d2 = d2 - e ** 2; d2[chosen] = -np.inf
        nxt = int(np.argmax(d2))
        if d2[nxt] < eps:
            break
        chosen.append(nxt)
    return chosen

check("R2 DPP", _t2)

In [ ]:
# ---------- R3 ----------
def kl_rows(p, Q, alpha=0.01):
    Qt = (1 - alpha) * Q + alpha * p; m = p > 0
    return (p[m] * np.log(p[m] / Qt[:, m])).sum(1)

def rerank_home(u, ids, rel, w, k=K):
    ids, rel = np.asarray(ids), np.asarray(rel)
    p, S = genre_dist_history(u), SIM(ids)
    chosen, acc = [], np.zeros(len(GENRES)); max_sim = np.zeros(len(ids)); avail = np.ones(len(ids), bool)
    for t in range(min(k, len(ids))):
        score = rel + w.get("nov", 0) * NOV[ids] + (w.get("div", 0) * (1 - max_sim) if t else 0)
        if w.get("cal", 0):
            Q = acc + P_gi[ids]; score = score - w["cal"] * kl_rows(p, Q / Q.sum(1, keepdims=True))
        score[~avail] = -np.inf
        j = int(np.argmax(score)); chosen.append(j); avail[j] = False
        acc += P_gi[ids[j]]; max_sim = np.maximum(max_sim, S[j])
    return list(ids[chosen])

base_test = evaluate_lists({u: list(CANDS[u][0][:K]) for u in eval_users})
pd.Series(base_test, name="baseline (test)").round(4)

In [ ]:
# ---------- R4 ----------
def pareto_mask(P):
    keep = np.ones(len(P), bool)
    for i in range(len(P)):
        keep[i] = not (np.all(P >= P[i], 1) & np.any(P > P[i], 1)).any()
    return keep

def choose_weights(grid, users):
    base = evaluate_lists({u: list(CANDS[u][0][:K]) for u in users})
    rows = [{**w, **evaluate_lists({u: rerank_home(u, *CANDS[u], w) for u in users})} for w in grid]
    df = pd.DataFrame(rows)
    df["pareto"] = pareto_mask(np.c_[df["NDCG@10"], df["ILD@10"], -df["KL"]])
    df["gain"] = df["ILD@10"] / base["ILD@10"] - df["KL"] / base["KL"]          # compromiso diversidad + calibración
    keep_ndcg = df["NDCG@10"] >= .93 * base["NDCG@10"]
    ok = df[keep_ndcg & (df["ILD@10"] >= 1.10 * base["ILD@10"]) & (df["KL"] <= .75 * base["KL"])]
    if len(ok):
        pick = ok.sort_values("NDCG@10", ascending=False).iloc[0]
    else:                                                                        # ε-restricción no factible
        pick = df[df.pareto & keep_ndcg].sort_values("gain", ascending=False).iloc[0]
    return df, {k: float(pick[k]) for k in ["div", "cal", "nov"]}, base

grid = [{"div": a, "cal": b, "nov": c} for a, b, c in itertools.product([0, .05, .1, .15, .3], [0, .15, .3, .5, 1], [0, .1])]
sweep, W, base_val = choose_weights(grid, val_users)
fig, ax = plt.subplots(figsize=(7, 4))
s = ax.scatter(sweep["ILD@10"], sweep["NDCG@10"], c=sweep["KL"], cmap="viridis_r", s=25)
f = sweep[sweep.pareto]; ax.scatter(f["ILD@10"], f["NDCG@10"], facecolors="none", edgecolors="r", s=80, label="Pareto 3D")
pk = sweep[(sweep["div"] == W["div"]) & (sweep["cal"] == W["cal"]) & (sweep["nov"] == W["nov"])]
ax.scatter(pk["ILD@10"], pk["NDCG@10"], marker="*", s=250, c="gold", ec="k", label=f"elegido {W}")
ax.scatter(base_val["ILD@10"], base_val["NDCG@10"], marker="X", s=120, c="k", label="baseline")
plt.colorbar(s, label="KL"); ax.set_xlabel("ILD@10"); ax.set_ylabel("NDCG@10"); ax.legend(fontsize=7)
ax.set_title(f"Barrido en VALIDACIÓN ({len(grid)} configuraciones)"); plt.show()

final = evaluate_lists({u: rerank_home(u, *CANDS[u], W) for u in eval_users})
rep = pd.DataFrame({"baseline": base_test, "re-ranker": final}).T
crit = {"NDCG ≥ 0,93×": final["NDCG@10"] >= .93 * base_test["NDCG@10"], "ILD ≥ 1,10×": final["ILD@10"] >= 1.10 * base_test["ILD@10"],
        "KL ≤ 0,75×": final["KL"] <= .75 * base_test["KL"]}
print("R3 (test):", {k: "✅" if v else "❌" for k, v in crit.items()}); rep.round(4)

In [ ]:
# ---------- R5 ----------
def user_rel_full(u):
    s = cf.user_emb([u])[0] @ cf.item_emb.T
    z = (s - s.mean()) / (s.std() + 1e-9); r = 1 / (1 + np.exp(-(z - 2))); r[hist_by_user[u]] = 0
    return r

def apply_row_rules(items, kids, max_per_genre=4):
    out, cnt = [], collections.Counter()
    for i in items:
        gs = movies.genre_list.iat[i]
        if kids and set(gs) & MATURE:
            continue
        if cnt[gs[0]] >= max_per_genre:
            continue
        out.append(i); cnt[gs[0]] += 1
    return out

def candidate_rows_home(u, kids=False, n=30):
    r = user_rel_full(u); seen = set(hist_by_user[u])
    top = lambda pool: [int(i) for i in sorted(pool, key=lambda i: -r[i]) if i not in seen][:n]
    ids, rel = CANDS[u] if u in CANDS else candidates(u)
    rows = {("top", "Top para ti"): rerank_home(u, ids, rel, W, k=n)}
    for g in np.argsort(-genre_dist_history(u))[:3]:
        rows[("genero", f"Porque te gusta {GENRES[g]}")] = top(np.where(G[:, g] > 0)[0])
    rows[("pop", "Tendencias")] = [int(i) for i in np.argsort(-pop) if i not in seen][:n]
    rows[("joya", "Joyas ocultas")] = top(np.where(pop <= np.median(pop[pop > 0]))[0])
    return {k: apply_row_rules(v, kids) for k, v in rows.items()}, r

def page_greedy(rows, r, type_penalty=.3):
    page, used, prev, rem = [], set(), None, dict(rows)
    for pos in range(min(N_ROWS, len(rows))):
        scored = []
        for key, items in rem.items():
            fresh = [i for i in items if i not in used][:N_COLS]
            v = sum(ATT[pos, c] * r[i] for c, i in enumerate(fresh)) - type_penalty * ATT[pos, 0] * (key[0] == prev)
            scored.append((v, key, fresh))
        v, key, fresh = max(scored, key=lambda x: x[0])
        page.append((key[1], fresh)); used |= set(fresh); prev = key[0]; rem.pop(key)
    return page

def build_home(u, kids=False):
    rows, r = candidate_rows_home(u, kids)
    return page_greedy(rows, r)

def page_independent(u, kids=False):
    rows, r = candidate_rows_home(u, kids)
    ranked = sorted(rows.items(), key=lambda kv: -np.mean([r[i] for i in kv[1][:VISIBLE]] or [0]))[:N_ROWS]
    return [(k[1], v[:N_COLS]) for k, v in ranked]

users_h = eval_users[:100]; kids_set = set(users_h[::4])
res_h = {"Filas independientes": [], "Greedy de página + reglas": []}; viol = 0
for u in users_h:
    r = user_rel_full(u); kids = u in kids_set
    res_h["Filas independientes"].append(page_value(page_independent(u, kids), r))
    pg = build_home(u, kids); res_h["Greedy de página + reglas"].append(page_value(pg, r))
    viol += kids * sum(bool(set(movies.genre_list.iat[i]) & MATURE) for _, it in pg for i in it)
hdf = pd.DataFrame({k: {"utilidad": np.mean([a for a, _ in v]), "duplicados": np.mean([b for _, b in v])} for k, v in res_h.items()}).T
ok5 = hdf.loc["Greedy de página + reglas", "duplicados"] == 0 and viol == 0 and \
      hdf.loc["Greedy de página + reglas", "utilidad"] >= hdf.loc["Filas independientes", "utilidad"]
print("R5:", "✅" if ok5 else "❌", f"· violaciones infantiles={viol}"); hdf.round(3)

In [ ]:
# Visualización de una home
u0 = users_h[0]; page = build_home(u0)
fig, ax = plt.subplots(figsize=(11, 3.6)); cmap = plt.get_cmap("tab20")
for pos, (title, items) in enumerate(page):
    for c, i in enumerate(items):
        g = GENRES.index(next((x for x in movies.genre_list.iat[i] if x in GENRES), "Drama"))
        ax.add_patch(plt.Rectangle((c, -pos), .92, .8, fc=cmap(g % 20), ec="w"))
        ax.text(c + .46, -pos + .4, movies.title.iat[i][:9], ha="center", va="center", fontsize=5.5)
    ax.text(-.2, -pos + .4, title[:30], ha="right", va="center", fontsize=8)
ax.set_xlim(-5.5, N_COLS); ax.set_ylim(-len(page) + .5, 1); ax.axis("off")
ax.set_title(f"Home de CineMatch del usuario {u0} (color = género principal)"); plt.show()

In [ ]:
# ---------- R6 · feedback loop: baseline vs re-ranker ----------
def ground_truth(dim=32, base_ctr=.03, temp=1.0):
    Xall = build_interaction_matrix(ratings, n_users, n_items)
    svd = TruncatedSVD(dim, random_state=SEED).fit(Xall)
    S = (Xall @ svd.components_.T) @ svd.components_
    Z = (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
    return 1 / (1 + np.exp(-(temp * np.clip(Z, -3, 3) + np.log(base_ctr / (1 - base_ctr)))))

def run_loop(use_reranker, rounds=10, sim_users=None, seed=SEED):
    g = np.random.default_rng(seed); us = np.array(sim_users)
    Xobs = X.copy().tolil(); exposure = np.zeros(n_items); hist = []
    for t in range(rounds):
        Xc = Xobs.tocsr(); svd = TruncatedSVD(32, random_state=SEED).fit(Xc)
        Sc = np.asarray((Xc[us] @ svd.components_.T) @ svd.components_); Sc[Xc[us].nonzero()] = -np.inf
        recs = []
        for row, u in enumerate(us):
            top = np.argpartition(-Sc[row], N_CAND)[:N_CAND]; top = top[np.argsort(-Sc[row, top])]
            sc = Sc[row, top]; rel = (sc - sc.min()) / (sc.max() - sc.min() + 1e-9)
            recs.append(rerank_home(u, top, rel, W) if use_reranker else list(top[:K]))
        recs = np.array(recs); np.add.at(exposure, recs.ravel(), 1)
        clicks = g.random(recs.shape) < P_TRUE[us[:, None], recs]
        for r_, c_ in zip(*np.nonzero(clicks)):
            Xobs[us[r_], recs[r_, c_]] = 1
        hist.append({"ronda": t + 1, "Gini": gini(exposure), "cobertura acumulada": (exposure > 0).mean(), "clics": clicks.sum()})
    return pd.DataFrame(hist)

P_TRUE = ground_truth()
sim_users = eval_users[: (150 if FAST_DEV_RUN else len(eval_users))]
loops = {"Baseline CF": run_loop(False, sim_users=sim_users), "Re-ranker home": run_loop(True, sim_users=sim_users)}
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
for name, h in loops.items():
    ax[0].plot(h.ronda, h.Gini, "o-", label=name); ax[1].plot(h.ronda, h["cobertura acumulada"], "o-", label=name)
    ax[2].plot(h.ronda, h.clics.cumsum(), "o-", label=name)
for a, t in zip(ax, ["Gini de exposición", "Cobertura acumulada del catálogo", "Clics acumulados"]):
    a.set_title(t); a.set_xlabel("ronda"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 🚀 Retos extra (nivel experto)
1. **DPP en la fila "Top para ti"**: sustituye el término de diversidad por un DPP con el kernel de YouTube y busca $(\alpha, \sigma)$ en validación. ¿Domina al greedy combinado en el frente de Pareto?
2. **Calibración por embeddings**: en vez de géneros, agrupa los ítems con k-means sobre embeddings de contenido (módulo 03) y calibra sobre esos *clusters*.
3. **Fairness amortizada de proveedores** dentro de la home: garantiza que la exposición acumulada del 90 % menos popular del catálogo sea ≥ 0,8 × su mérito, a lo largo de todos los usuarios.
4. **Pesos personalizados**: aprende $w_{div}$ por segmento de usuario (p. ej., por entropía de su historial). ¿Mejora el frente?
5. **Off-policy**: loguea propensiones del re-ranker (con algo de aleatoriedad) y estima con IPS el CTR de otra configuración de pesos (módulo 14).

## 🤔 Reflexión (producción / MLOps)
- ¿Qué métricas *beyond-accuracy* pondrías como **guardrails** en un A/B test de este re-ranker (módulo 15)? ¿Cuánto tiempo lo dejarías correr y por qué?
- ¿Dónde vive este re-ranker en la arquitectura online (módulo 16)? ¿Qué presupuesto de latencia le das y qué pasa si se excede (fallback)?
- ¿Qué logs necesitas para que el siguiente modelo no aprenda tus reglas como si fueran preferencias?
- Si el negocio cambia los pesos cada semana, ¿cómo versionas y auditas esas decisiones (módulo 17)?